# IDSA workbench experiment results

This notebook reads the recorded outputs from `experiment_plan.py`. From the repository root, run `python experiment_plan.py` to recompute them. The experiment only uses the repository's October 12 snapshot and validates on 2023, 2024, and 2025. All groups use frozen validation history and strict earlier-day training features. The initial fold uses 0.50. Later thresholds and any mixture weights use only earlier validation folds. Rank by the mean of 2024 and 2025.

Workbench D0 and M0–M12 correspond to the original plan. A0 is the first round's HistGradientBoosting M1 baseline. A1–A5 test history features and cold-start decisions against A0. M9 reports coverage with the composite key, while its subgroup scores use the license-only split for comparison.

In [1]:
import gzip
import json
from pathlib import Path

import pandas as pd
from experiment_plan import PLAN

with gzip.open(Path('experiment_plan_results.json.gz'), 'rt', encoding='utf-8') as snapshot:
    folds = json.load(snapshot)['folds']
def get_scores(cohort):
    return pd.DataFrame([
        {'code': f['code'], 'fold': int(f['fold']), **f['subgroups'][cohort]}
        for f in folds if cohort in f['subgroups']
    ])

history = get_scores('all')
scores = history.pivot(index='code', columns='fold', values='macro_f1')
scores['mean_2024_25'] = scores[[2024, 2025]].mean(axis=1)
scores = scores.sort_values('mean_2024_25', ascending=False)
scores['description'] = [PLAN[code][1] for code in scores.index]
scores.round(4)

fold    2023    2024    2025  mean_2024_25         description
code                                                          
M4    0.4665  0.5808  0.5866        0.5837             流程与检查间隔
M12   0.4665  0.5795  0.5866        0.5830           验证折学习融合权重
A0    0.4539  0.5797  0.5863        0.5830  第一轮 Boosting M1 对照
A3    0.4539  0.5797  0.5863        0.5830           无匹配历史单独阈值
M3    0.4522  0.5797  0.5861        0.5829              加入平滑历史
M9    0.4637  0.5804  0.5837        0.5820           许可证＋地址的历史
M8    0.4663  0.5795  0.5825        0.5810          固定 M4 特征对照
A5    0.4541  0.5774  0.5838        0.5806        上次失败且本次复查的间隔
M7    0.4668  0.5785  0.5824        0.5805            最近 5 年拟合
A1    0.4558  0.5744  0.5850        0.5797           门店历史按复查拆分
A2    0.4533  0.5755  0.5820        0.5787           按当前检查类型平滑
M6    0.4751  0.5743  0.5825        0.5784            最近 3 年拟合
M11   0.4690  0.5747  0.5812        0.5779          复查与非复查独立拟合
A4    0.4539  0.5748  0.5781        0.5764          无匹配

In [2]:
cohorts = ['cold_start', 'invalid_license', 'valid_no_history',
           'history_1_2', 'history_3_plus', 'reinspection', 'non_reinspection']
cohort_scores = pd.concat(
    [get_scores(c).assign(cohort=c) for c in cohorts],
    ignore_index=True,
)
(cohort_scores.query("fold >= 2024")
 .pivot_table(index='code', columns='cohort', values='macro_f1')
 .reindex(scores.index)
 .round(4))

cohort  cold_start  history_1_2  history_3_plus  invalid_license  non_reinspection  reinspection  valid_no_history
code                                                                                                              
M4          0.5410       0.5659          0.5928           0.6304            0.5707        0.4760            0.5382
M12         0.5380       0.5737          0.5914           0.6304            0.5702        0.4792            0.5351
A0          0.5443       0.5816          0.5895           0.6418            0.5695        0.4730            0.5415
A3          0.5443       0.5816          0.5895           0.6418            0.5695        0.4730            0.5415
M3          0.5543       0.5912          0.5864           0.6090            0.5644        0.4847            0.5526
M9          0.5534       0.5860          0.5858           0.6252            0.5620        0.4764            0.5515
M8          0.5379       0.5661          0.5888           0.6340            0.56

In [3]:
decisions = pd.DataFrame([{k: v for k, v in f.items() if k != 'subgroups'} for f in folds])
decisions.loc[decisions.code.isin(['A3', 'A4', 'M9', 'M12']),
              ['code', 'fold', 'threshold', 'cold_threshold', 'composite_history_count',
               'lightgbm_weight', 'earlier_unique_correct']]

   code  fold  threshold  cold_threshold  composite_history_count  lightgbm_weight earlier_unique_correct
10   M9  2023       0.50             NaN                  12936.0              NaN                    NaN
13  M12  2023       0.50             NaN                      NaN              1.0                   None
17   A3  2023       0.50            0.50                      NaN              NaN                    NaN
18   A4  2023       0.50            0.50                      NaN              NaN                    NaN
30   M9  2024       0.28             NaN                  13643.0              NaN                    NaN
33  M12  2024       0.30             NaN                      NaN              0.5             [508, 489]
37   A3  2024       0.30            0.30                      NaN              NaN                    NaN
38   A4  2024       0.30            0.24                      NaN              NaN                    NaN
50   M9  2025       0.28             NaN      

## What the rolling folds support

- M4 leads with a 2024–2025 mean Macro-F1 of 0.5837. M3 scores 0.5829 and the previous A0 baseline scores 0.5830. These margins are small. M3 has stronger cold-start mean Macro-F1 (0.5543 vs 0.5410 for M4).
- Under the LightGBM sequence, adding raw license history raises the mean from 0.5567 to 0.5717. Adding the smoothed rate raises it to 0.5829. M4 adds both the process features and intervals, so their individual effects cannot be inferred from the M3–M4 difference.
- Decay and shorter fit windows lag M4. CatBoost and the validation-selected blend do not exceed it under this sweep. The blend uses 0.5 LightGBM weight for 2024 and returns to 1.0 for 2025.
- The cold-start-specific threshold selected 0.30 on both scored folds, leaving A0 predictions unchanged. The tested Logistic fallback increased cold-start recall but overflagged enough non-failures to lower mean subgroup Macro-F1 from 0.5443 to 0.4413.
- These are model-development folds. This sweep used no 2026 outcome records; July–October 2026 remains reserved for local holdout evaluation.